In [1]:
import os
import pickle
import math
import pandas as pd
from collections import Counter

In [2]:
K = 0.3

assignment_4_path = "../Assignment_4"

models_path = os.path.join(assignment_4_path, "models")
data_path = os.path.join(assignment_4_path, "data")

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [3]:
print(os.listdir(models_path))
print(os.listdir(data_path))

['bigram_context_counts.pkl', 'bigram_counts.pkl', 'bigram_laplace.pkl', 'bigram_probabilities.pkl', 'quadrigram_context_counts.pkl', 'quadrigram_counts.pkl', 'quadrigram_laplace.pkl', 'quadrigram_probabilities.pkl', 'trigram_context_counts.pkl', 'trigram_counts.pkl', 'trigram_laplace.pkl', 'trigram_probabilities.pkl', 'unigram_counts.pkl', 'unigram_laplace.pkl', 'unigram_probabilities.pkl', 'vocabulary.pkl']
['dev.parquet', 'test.parquet', 'train.parquet']


In [4]:
with open(os.path.join(models_path, "unigram_counts.pkl"), "rb") as f:
    unigram_counts = pickle.load(f)

with open(os.path.join(models_path, "bigram_counts.pkl"), "rb") as f:
    bigram_counts = pickle.load(f)

with open(os.path.join(models_path, "trigram_counts.pkl"), "rb") as f:
    trigram_counts = pickle.load(f)

with open(os.path.join(models_path, "quadrigram_counts.pkl"), "rb") as f:
    quadrigram_counts = pickle.load(f)

with open(os.path.join(models_path, "vocabulary.pkl"), "rb") as f:
    vocabulary = pickle.load(f)

In [5]:
train_df = pd.read_parquet(
    os.path.join(data_path, "train.parquet")
)

dev_df = pd.read_parquet(
    os.path.join(data_path, "dev.parquet")
)

test_df = pd.read_parquet(
    os.path.join(data_path, "test.parquet")
)

print("Training sentences:", len(train_df))
print("Development sentences:", len(dev_df))
print("Test sentences:", len(test_df))

Training sentences: 998000
Development sentences: 1000
Test sentences: 1000


In [6]:
V = len(vocabulary)

total_unigram_tokens = sum(unigram_counts.values())

print("K =", K)
print("Vocabulary size =", V)
print("Total unigram tokens =", total_unigram_tokens)

K = 0.3
Vocabulary size = 307861
Total unigram tokens = 24079766


In [7]:
bigram_context_counts = Counter()

for (w1, w2), count in bigram_counts.items():
    bigram_context_counts[w1] += count


trigram_context_counts = Counter()

for (w1, w2, w3), count in trigram_counts.items():
    trigram_context_counts[(w1, w2)] += count


quadrigram_context_counts = Counter()

for (w1, w2, w3, w4), count in quadrigram_counts.items():
    quadrigram_context_counts[(w1, w2, w3)] += count

In [8]:
def add_k_unigram(word):
    count = unigram_counts.get((word,), 0)
    numerator = count + K
    denominator = total_unigram_tokens + (K * V)
    return numerator / denominator

In [9]:
def add_k_bigram(w1, w2):
    count = bigram_counts.get((w1, w2), 0)
    context_count = bigram_context_counts.get(w1, 0)

    numerator = count + K
    denominator = context_count + (K * V)

    return numerator / denominator

In [10]:
def add_k_trigram(w1, w2, w3):
    count = trigram_counts.get((w1, w2, w3), 0)
    context_count = trigram_context_counts.get((w1, w2), 0)

    numerator = count + K
    denominator = context_count + (K * V)

    return numerator / denominator

In [11]:
def add_k_quadrigram(w1, w2, w3, w4):
    count = quadrigram_counts.get((w1, w2, w3, w4), 0)
    context_count = quadrigram_context_counts.get((w1, w2, w3), 0)

    numerator = count + K
    denominator = context_count + (K * V)

    return numerator / denominator

In [12]:
print("Unigram:", add_k_unigram("भारत"))

print("Bigram:", add_k_bigram("भारत", "में"))

print("Trigram:", add_k_trigram("भारत", "में", "लोग"))

print(
    "Quadrigram:",
    add_k_quadrigram("भारत", "में", "लोग", "रहते")
)

Unigram: 0.0011516695700592603
Bigram: 0.042657719081203
Trigram: 0.00013643082598094276
Quadrigram: 3.2477620213204747e-06


In [13]:
print("Unseen unigram:", add_k_unigram("xyzabc"))

print(
    "Unseen bigram:",
    add_k_bigram("xyzabc", "qwerty")
)

print(
    "Unseen trigram:",
    add_k_trigram("xyzabc", "qwerty", "asdfgh")
)

print(
    "Unseen quadrigram:",
    add_k_quadrigram(
        "xyzabc",
        "qwerty",
        "asdfgh",
        "zxcvbn"
    )
)

Unseen unigram: 1.2410990290993993e-08
Unseen bigram: 3.2482191638434227e-06
Unseen trigram: 3.2482191638434227e-06
Unseen quadrigram: 3.2482191638434227e-06


In [14]:
unigram_add_k_probabilities = {}

for word in vocabulary:
    unigram_add_k_probabilities[word] = add_k_unigram(word)

print("Number of unigram probabilities:", len(unigram_add_k_probabilities))

Number of unigram probabilities: 307861


In [15]:
bigram_add_k_probabilities = {}

for (w1, w2) in bigram_counts:
    bigram_add_k_probabilities[(w1, w2)] = add_k_bigram(w1, w2)

print("Number of bigram probabilities:", len(bigram_add_k_probabilities))

Number of bigram probabilities: 3782573


In [16]:
trigram_add_k_probabilities = {}

for (w1, w2, w3) in trigram_counts:
    trigram_add_k_probabilities[(w1, w2, w3)] = add_k_trigram(
        w1,
        w2,
        w3
    )

print("Number of trigram probabilities:", len(trigram_add_k_probabilities))

Number of trigram probabilities: 10720662


In [17]:
quadrigram_add_k_probabilities = {}

for (w1, w2, w3, w4) in quadrigram_counts:
    quadrigram_add_k_probabilities[(w1, w2, w3, w4)] = add_k_quadrigram(
        w1,
        w2,
        w3,
        w4
    )

print(
    "Number of quadrigram probabilities:",
    len(quadrigram_add_k_probabilities)
)

Number of quadrigram probabilities: 15632208


In [18]:
with open("models/unigram_add_k.pkl", "wb") as f:
    pickle.dump(unigram_add_k_probabilities, f)

with open("models/bigram_add_k.pkl", "wb") as f:
    pickle.dump(bigram_add_k_probabilities, f)

with open("models/trigram_add_k.pkl", "wb") as f:
    pickle.dump(trigram_add_k_probabilities, f)

with open("models/quadrigram_add_k.pkl", "wb") as f:
    pickle.dump(quadrigram_add_k_probabilities, f)

In [19]:
def prepare_sentence(tokens):
    return ["<s>"] + list(tokens) + ["</s>"]


dev_sentences = [
    prepare_sentence(tokens)
    for tokens in dev_df["tokens"]
]

test_sentences = [
    prepare_sentence(tokens)
    for tokens in test_df["tokens"]
]

print(dev_sentences[0])
print(test_sentences[0])

['<s>', 'हमने', 'अपनी', 'जांच', 'पर', 'ध्यान', 'केंद्रित', 'किया', '।', '</s>']
['<s>', 'सूत्रों', 'ने', 'जानकारी', 'दी', 'है', 'कि', 'उपचुनावों', 'को', 'लेकर', 'भाजपा', 'और', 'सपा', 'में', 'समझौता', 'हो', 'गया', 'है', ',', 'इसीलिए', 'सपा', 'की', 'तरफ', 'से', 'नोएडा', 'में', 'कोई', 'कैंडिडेट', 'अभी', 'नहीं', 'उतारा', 'गया', 'है', '।', '</s>']


In [20]:
def evaluate_unigram(sentences):
    total_log_probability = 0.0
    total_tokens = 0

    for sentence in sentences:
        for word in sentence:
            probability = add_k_unigram(word)
            total_log_probability += math.log2(probability)
            total_tokens += 1

    cross_entropy = -total_log_probability / total_tokens
    perplexity = 2 ** cross_entropy

    return total_log_probability, total_tokens, cross_entropy, perplexity

In [21]:
def evaluate_bigram(sentences):
    total_log_probability = 0.0
    total_tokens = 0

    for sentence in sentences:
        for i in range(1, len(sentence)):
            w1 = sentence[i - 1]
            w2 = sentence[i]

            probability = add_k_bigram(w1, w2)

            total_log_probability += math.log2(probability)
            total_tokens += 1

    cross_entropy = -total_log_probability / total_tokens
    perplexity = 2 ** cross_entropy

    return total_log_probability, total_tokens, cross_entropy, perplexity

In [22]:
def evaluate_trigram(sentences):
    total_log_probability = 0.0
    total_tokens = 0

    for sentence in sentences:
        for i in range(2, len(sentence)):
            w1 = sentence[i - 2]
            w2 = sentence[i - 1]
            w3 = sentence[i]

            probability = add_k_trigram(w1, w2, w3)

            total_log_probability += math.log2(probability)
            total_tokens += 1

    cross_entropy = -total_log_probability / total_tokens
    perplexity = 2 ** cross_entropy

    return total_log_probability, total_tokens, cross_entropy, perplexity

In [23]:
def evaluate_quadrigram(sentences):
    total_log_probability = 0.0
    total_tokens = 0

    for sentence in sentences:
        for i in range(3, len(sentence)):
            w1 = sentence[i - 3]
            w2 = sentence[i - 2]
            w3 = sentence[i - 1]
            w4 = sentence[i]

            probability = add_k_quadrigram(
                w1,
                w2,
                w3,
                w4
            )

            total_log_probability += math.log2(probability)
            total_tokens += 1

    cross_entropy = -total_log_probability / total_tokens
    perplexity = 2 ** cross_entropy

    return total_log_probability, total_tokens, cross_entropy, perplexity

In [24]:
dev_unigram = evaluate_unigram(dev_sentences)

dev_bigram = evaluate_bigram(dev_sentences)

dev_trigram = evaluate_trigram(dev_sentences)

dev_quadrigram = evaluate_quadrigram(dev_sentences)

In [25]:
dev_results = pd.DataFrame({
    "Model": [
        "Unigram",
        "Bigram",
        "Trigram",
        "Quadrigram"
    ],
    "Log Probability": [
        dev_unigram[0],
        dev_bigram[0],
        dev_trigram[0],
        dev_quadrigram[0]
    ],
    "Tokens": [
        dev_unigram[1],
        dev_bigram[1],
        dev_trigram[1],
        dev_quadrigram[1]
    ],
    "Cross Entropy": [
        dev_unigram[2],
        dev_bigram[2],
        dev_trigram[2],
        dev_quadrigram[2]
    ],
    "Perplexity": [
        dev_unigram[3],
        dev_bigram[3],
        dev_trigram[3],
        dev_quadrigram[3]
    ]
})

dev_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-260298.199386,24954,10.431121,1380.639776
1,Bigram,-254959.215409,23954,10.643701,1599.827975
2,Trigram,-327767.063661,22954,14.279300,19883.725886
3,Quadrigram,-363420.409855,21954,16.553722,96198.172487


In [26]:
test_unigram = evaluate_unigram(test_sentences)

test_bigram = evaluate_bigram(test_sentences)

test_trigram = evaluate_trigram(test_sentences)

test_quadrigram = evaluate_quadrigram(test_sentences)

In [27]:
test_results = pd.DataFrame({
    "Model": [
        "Unigram",
        "Bigram",
        "Trigram",
        "Quadrigram"
    ],
    "Log Probability": [
        test_unigram[0],
        test_bigram[0],
        test_trigram[0],
        test_quadrigram[0]
    ],
    "Tokens": [
        test_unigram[1],
        test_bigram[1],
        test_trigram[1],
        test_quadrigram[1]
    ],
    "Cross Entropy": [
        test_unigram[2],
        test_bigram[2],
        test_trigram[2],
        test_quadrigram[2]
    ],
    "Perplexity": [
        test_unigram[3],
        test_bigram[3],
        test_trigram[3],
        test_quadrigram[3]
    ]
})

test_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-286713.464298,27068,10.592340,1543.874954
1,Bigram,-281159.412105,26068,10.785615,1765.198867
2,Trigram,-360701.307972,25068,14.388914,21453.337298
3,Quadrigram,-399666.815339,24068,16.605734,99729.617169


In [28]:
print("Development Set Results")
print()

for _, row in dev_results.iterrows():
    print(row["Model"])
    print(f"Log Probability: {row['Log Probability']:.4f}")
    print(f"Cross Entropy: {row['Cross Entropy']:.4f}")
    print(f"Perplexity: {row['Perplexity']:.4f}")
    print()

Development Set Results

Unigram
Log Probability: -260298.1994
Cross Entropy: 10.4311
Perplexity: 1380.6398

Bigram
Log Probability: -254959.2154
Cross Entropy: 10.6437
Perplexity: 1599.8280

Trigram
Log Probability: -327767.0637
Cross Entropy: 14.2793
Perplexity: 19883.7259

Quadrigram
Log Probability: -363420.4099
Cross Entropy: 16.5537
Perplexity: 96198.1725



In [29]:
print("Test Set Results")
print()

for _, row in test_results.iterrows():
    print(row["Model"])
    print(f"Log Probability: {row['Log Probability']:.4f}")
    print(f"Cross Entropy: {row['Cross Entropy']:.4f}")
    print(f"Perplexity: {row['Perplexity']:.4f}")
    print()

Test Set Results

Unigram
Log Probability: -286713.4643
Cross Entropy: 10.5923
Perplexity: 1543.8750

Bigram
Log Probability: -281159.4121
Cross Entropy: 10.7856
Perplexity: 1765.1989

Trigram
Log Probability: -360701.3080
Cross Entropy: 14.3889
Perplexity: 21453.3373

Quadrigram
Log Probability: -399666.8153
Cross Entropy: 16.6057
Perplexity: 99729.6172



In [30]:
dev_results.to_csv(
    "results/development_results.csv",
    index=False
)

test_results.to_csv(
    "results/test_results.csv",
    index=False
)

In [31]:
dev_results_final = dev_results.copy()
dev_results_final["Dataset"] = "Development"

test_results_final = test_results.copy()
test_results_final["Dataset"] = "Test"

all_results = pd.concat(
    [
        dev_results_final,
        test_results_final
    ],
    ignore_index=True
)

all_results = all_results[
    [
        "Dataset",
        "Model",
        "Log Probability",
        "Tokens",
        "Cross Entropy",
        "Perplexity"
    ]
]

all_results

,Dataset,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Development,Unigram,-260298.199386,24954,10.431121,1380.639776
1,Development,Bigram,-254959.215409,23954,10.643701,1599.827975
2,Development,Trigram,-327767.063661,22954,14.279300,19883.725886
3,Development,Quadrigram,-363420.409855,21954,16.553722,96198.172487
4,Test,Unigram,-286713.464298,27068,10.592340,1543.874954
5,Test,Bigram,-281159.412105,26068,10.785615,1765.198867
6,Test,Trigram,-360701.307972,25068,14.388914,21453.337298
7,Test,Quadrigram,-399666.815339,24068,16.605734,99729.617169


In [32]:
all_results.to_csv(
    "results/add_k_results.csv",
    index=False
)

In [33]:
print("Assignment 5 completed")
print()
print("Smoothing method: Add-K")
print("K =", K)
print("Vocabulary size:", V)
print()
print("Development sentences:", len(dev_df))
print("Test sentences:", len(test_df))
print()
print("Models:")
print("1. Unigram")
print("2. Bigram")
print("3. Trigram")
print("4. Quadrigram")

Assignment 5 completed

Smoothing method: Add-K
K = 0.3
Vocabulary size: 307861

Development sentences: 1000
Test sentences: 1000

Models:
1. Unigram
2. Bigram
3. Trigram
4. Quadrigram
